# OOD probe: ECHR (legal briefs, Poudyal et al. 2020)Runs v4 on 20 European Court of Human Rights case briefs with a proxy gold derived from ECHR's agent labels. See Section 6.3 of the paper. Uses the salvage parser fallback (Section 7) because long inputs trigger the relation-loop pathology on this corpus.

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
python3 <<'PY'
import sys, json, random, time
sys.path.insert(0, '.')
from pathlib import Path
from src.phase2.config import load_phase2_config
from src.phase2.student import build_student

cfg = load_phase2_config('configs/phase2_beta_qwen1.5b_lora_v4.yaml')
cfg.student.max_input_len = 2048
cfg.student.max_target_len = 2048

student = build_student(cfg.student)
student.load(cfg.student_output_dir)
print(f"loaded v4  |  domain: LEGAL (ECHR — completely unseen)\n", flush=True)

echr_dir = Path('phase2_data/raw/echr/extracted/all-data')
files = sorted(echr_dir.glob('*.json'))
print(f"ECHR corpus: {len(files)} case files", flush=True)

random.seed(42)
sample = random.sample(files, 5)

for idx, fp in enumerate(sample, 1):
    with open(fp) as f:
        case = json.load(f)
    # ECHR schema: 'all_arguments' is a list of {text, agent, ...} dicts
    args = case.get('all_arguments', [])
    if not args: continue
    # Concatenate the first ~5 arguments as our test text
    text = ' '.join(a.get('text', '') for a in args[:5])[:3000]

    print(f"\n{'='*70}\nCASE {idx}/{len(sample)}: {fp.stem}  ({len(text)} chars)")
    print(f"Input snippet: {text[:180]}...")

    t0 = time.time()
    pred, reasoning = student.predict(text)
    elapsed = time.time() - t0

    n_c = len(pred['claim_components'])
    n_p = len(pred['premise_components'])
    n_r = len(pred['relations'])

    status = 'REAL' if (n_c + n_p) > 0 else 'EMPTY'
    print(f"[{status}]  time={elapsed:.0f}s  reasoning_chars={len(reasoning)}")
    print(f"  claims: {n_c}  premises: {n_p}  relations: {n_r}")
    if n_c > 0:
        print(f"  first claim: {pred['claim_components'][0]['text'][:120]}")
    if n_p > 0:
        print(f"  first premise: {pred['premise_components'][0]['text'][:120]}")
PY

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
python3 <<'PY'
import json, random
from pathlib import Path
from collections import Counter

echr_dir = Path('phase2_data/raw/echr/extracted/all-data')
files = sorted(echr_dir.glob('*.json'))
random.seed(7)
sample = random.sample(files, 3)

for fp in sample:
    with open(fp) as f:
        case = json.load(f)
    print(f"\n=== {fp.name} ===")
    print(f"top-level keys: {list(case.keys())}")

    # Probe all_arguments schema
    args = case.get('all_arguments', [])
    print(f"n_arguments: {len(args)}")
    if args:
        print(f"first arg keys: {list(args[0].keys())}")
        print(f"first arg sample: {json.dumps(args[0], ensure_ascii=False)[:400]}")

        # Count field distributions
        agent_ct = Counter(a.get('agent', '?') for a in args)
        type_ct  = Counter(a.get('type',  '?') for a in args)
        print(f"agent distribution: {dict(agent_ct)}")
        print(f"type distribution: {dict(type_ct)}")

    # Look for other gold-signal keys
    for k in ('relations', 'links', 'arguments', 'premises', 'conclusions', 'gold'):
        if k in case:
            v = case[k]
            print(f"has '{k}': {type(v).__name__}, len={len(v) if hasattr(v,'__len__') else '?'}")
            if isinstance(v, list) and v:
                print(f"  first item: {json.dumps(v[0], ensure_ascii=False)[:300]}")
PY

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
python3 <<'PY'
import sys, json, random, time, re
sys.path.insert(0, '.')
from pathlib import Path
from src.phase2.config import load_phase2_config
from src.phase2.student import build_student

cfg = load_phase2_config('configs/phase2_beta_qwen1.5b_lora_v4.yaml')
cfg.student.max_input_len  = 2048
cfg.student.max_target_len = 768   # v4 doesn't CoT on legal; keep tight

student = build_student(cfg.student)
student.load(cfg.student_output_dir)
print("loaded v4\n", flush=True)

echr_dir = Path('phase2_data/raw/echr/extracted/all-data')
files    = sorted(echr_dir.glob('*.json'))
random.seed(42)
sample = random.sample(files, 20)

def agent_to_role(a):
    if a == 'ECHR': return 'claim'
    if a in ('Applicant', 'State', 'Third Parties'): return 'premise'
    return None

def toks(s): return set(re.findall(r'\w+', s.lower()))
def overlap(a, b):
    ta, tb = toks(a), toks(b)
    return (len(ta & tb) / len(ta | tb)) if (ta and tb) else 0.0

tp_c=fp_c=fn_c=tp_p=fp_p=fn_p=n_real=0
n_extracted=n_gold_total=0
t_start = time.time()

for idx, fp in enumerate(sample, 1):
    case = json.load(open(fp))
    args = case.get('all_arguments', [])
    gold = [{'text': a['text'], 'role': agent_to_role(a.get('agent'))}
            for a in args if agent_to_role(a.get('agent')) and a.get('text')]
    if not gold: continue

    text = ' '.join(a['text'] for a in args)[:3000]
    t0 = time.time()
    try:
        pred, _ = student.predict(text)
    except Exception as e:
        print(f"[{idx}/{len(sample)}] ERR {e}", flush=True); continue
    dt = time.time()-t0

    pred_spans = ([{'text':c['text'],'role':'claim'}   for c in pred['claim_components']] +
                  [{'text':p['text'],'role':'premise'} for p in pred['premise_components']])
    gc = sum(1 for g in gold if g['role']=='claim')
    gp = len(gold) - gc

    if not pred_spans:
        print(f"[{idx}/{len(sample)}] {fp.stem}: EMPTY  t={dt:.0f}s  gold_c/p={gc}/{gp}", flush=True)
        fn_c += gc; fn_p += gp
        n_gold_total += len(gold)
        continue

    matched_g = [False]*len(gold)
    matched_p = [False]*len(pred_spans)
    for pi, ps in enumerate(pred_spans):
        best_gi, best_ov = -1, 0.0
        for gi, gs in enumerate(gold):
            if matched_g[gi]: continue
            ov = overlap(ps['text'], gs['text'])
            if ov > best_ov: best_gi, best_ov = gi, ov
        if best_ov >= 0.5:
            matched_g[best_gi] = True
            matched_p[pi] = True
            if ps['role'] == gold[best_gi]['role']:
                if ps['role']=='claim': tp_c+=1
                else: tp_p+=1
            else:
                if ps['role']=='claim': fp_c+=1
                else: fp_p+=1
                if gold[best_gi]['role']=='claim': fn_c+=1
                else: fn_p+=1
    for pi,m in enumerate(matched_p):
        if not m:
            if pred_spans[pi]['role']=='claim': fp_c+=1
            else: fp_p+=1
    for gi,m in enumerate(matched_g):
        if not m:
            if gold[gi]['role']=='claim': fn_c+=1
            else: fn_p+=1

    n_extracted += sum(matched_p)
    n_gold_total += len(gold)
    n_real += 1
    print(f"[{idx}/{len(sample)}] {fp.stem}: pred_c/p={len(pred['claim_components'])}/{len(pred['premise_components'])} "
          f"gold_c/p={gc}/{gp}  t={dt:.0f}s", flush=True)

def f1(tp,fp,fn):
    p=tp/max(tp+fp,1); r=tp/max(tp+fn,1)
    return 2*p*r/max(p+r,1e-6), p, r

f1c,pc,rc = f1(tp_c,fp_c,fn_c)
f1p,pp,rp = f1(tp_p,fp_p,fn_p)
print(f"\n=== ECHR RESULTS (non-empty {n_real}/{len(sample)}) ===")
print(f"claim   F1={f1c:.3f}  P={pc:.3f}  R={rc:.3f}  (tp={tp_c} fp={fp_c} fn={fn_c})")
print(f"premise F1={f1p:.3f}  P={pp:.3f}  R={rp:.3f}  (tp={tp_p} fp={fp_p} fn={fn_p})")
print(f"COMPONENT F1 (macro): {(f1c+f1p)/2:.3f}")
print(f"extraction rate: {n_extracted}/{n_gold_total} = {n_extracted/max(n_gold_total,1):.1%}")
print(f"wall: {(time.time()-t_start)/60:.1f} min")
PY

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
mkdir -p eval_logs

nohup python3 -u <<'PY' > eval_logs/echr_v4_2048.log 2>&1 &
import sys, json, random, time, re
sys.path.insert(0, '.')
from pathlib import Path
from src.phase2.config import load_phase2_config
from src.phase2.student import build_student

cfg = load_phase2_config('configs/phase2_beta_qwen1.5b_lora_v4.yaml')
cfg.student.max_input_len  = 2048
cfg.student.max_target_len = 2048   # full budget this time

student = build_student(cfg.student)
student.load(cfg.student_output_dir)
print("loaded v4  |  max_target=2048\n", flush=True)

echr_dir = Path('phase2_data/raw/echr/extracted/all-data')
files    = sorted(echr_dir.glob('*.json'))
random.seed(42)
sample = random.sample(files, 20)

def agent_to_role(a):
    if a == 'ECHR': return 'claim'
    if a in ('Applicant', 'State', 'Third Parties'): return 'premise'
    return None

def toks(s): return set(re.findall(r'\w+', s.lower()))
def overlap(a, b):
    ta, tb = toks(a), toks(b)
    return (len(ta & tb) / len(ta | tb)) if (ta and tb) else 0.0

tp_c=fp_c=fn_c=tp_p=fp_p=fn_p=n_real=0
n_extracted=n_gold_total=0
t_start = time.time()

for idx, fp in enumerate(sample, 1):
    case = json.load(open(fp))
    args = case.get('all_arguments', [])
    gold = [{'text': a['text'], 'role': agent_to_role(a.get('agent'))}
            for a in args if agent_to_role(a.get('agent')) and a.get('text')]
    if not gold: continue
    text = ' '.join(a['text'] for a in args)[:3000]
    t0 = time.time()
    try:
        pred, reasoning = student.predict(text)
    except Exception as e:
        print(f"[{idx}/{len(sample)}] ERR {e}", flush=True); continue
    dt = time.time()-t0
    pred_spans = ([{'text':c['text'],'role':'claim'}   for c in pred['claim_components']] +
                  [{'text':p['text'],'role':'premise'} for p in pred['premise_components']])
    gc = sum(1 for g in gold if g['role']=='claim')
    gp = len(gold) - gc
    if not pred_spans:
        print(f"[{idx}/{len(sample)}] {fp.stem}: EMPTY  t={dt:.0f}s  reason_chars={len(reasoning)}  gold_c/p={gc}/{gp}", flush=True)
        fn_c += gc; fn_p += gp
        n_gold_total += len(gold); continue
    matched_g = [False]*len(gold)
    matched_p = [False]*len(pred_spans)
    for pi, ps in enumerate(pred_spans):
        best_gi, best_ov = -1, 0.0
        for gi, gs in enumerate(gold):
            if matched_g[gi]: continue
            ov = overlap(ps['text'], gs['text'])
            if ov > best_ov: best_gi, best_ov = gi, ov
        if best_ov >= 0.5:
            matched_g[best_gi] = True; matched_p[pi] = True
            if ps['role'] == gold[best_gi]['role']:
                if ps['role']=='claim': tp_c+=1
                else: tp_p+=1
            else:
                if ps['role']=='claim': fp_c+=1
                else: fp_p+=1
                if gold[best_gi]['role']=='claim': fn_c+=1
                else: fn_p+=1
    for pi,m in enumerate(matched_p):
        if not m:
            if pred_spans[pi]['role']=='claim': fp_c+=1
            else: fp_p+=1
    for gi,m in enumerate(matched_g):
        if not m:
            if gold[gi]['role']=='claim': fn_c+=1
            else: fn_p+=1
    n_extracted += sum(matched_p); n_gold_total += len(gold); n_real += 1
    print(f"[{idx}/{len(sample)}] {fp.stem}: pred_c/p={len(pred['claim_components'])}/{len(pred['premise_components'])} "
          f"gold_c/p={gc}/{gp}  reason_chars={len(reasoning)}  t={dt:.0f}s", flush=True)

def f1(tp,fp,fn):
    p=tp/max(tp+fp,1); r=tp/max(tp+fn,1)
    return 2*p*r/max(p+r,1e-6), p, r

f1c,pc,rc = f1(tp_c,fp_c,fn_c)
f1p,pp,rp = f1(tp_p,fp_p,fn_p)
print(f"\n=== ECHR RESULTS @ max_target=2048 (non-empty {n_real}/{len(sample)}) ===")
print(f"claim   F1={f1c:.3f}  P={pc:.3f}  R={rc:.3f}  (tp={tp_c} fp={fp_c} fn={fn_c})")
print(f"premise F1={f1p:.3f}  P={pp:.3f}  R={rp:.3f}  (tp={tp_p} fp={fp_p} fn={fn_p})")
print(f"COMPONENT F1 (macro): {(f1c+f1p)/2:.3f}")
print(f"extraction rate: {n_extracted}/{n_gold_total} = {n_extracted/max(n_gold_total,1):.1%}")
print(f"wall: {(time.time()-t_start)/60:.1f} min")
PY

echo "PID: $!"
sleep 2
echo "kicked off — log at eval_logs/echr_v4_2048.log"
tail -5 eval_logs/echr_v4_2048.log

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
python3 <<'PY'
import sys
sys.path.insert(0, '.')
from src.phase2.config import load_phase2_config
from src.phase2.student import build_student

cfg = load_phase2_config('configs/phase2_beta_qwen1.5b_lora_v4.yaml')
student = build_student(cfg.student)
student.load(cfg.student_output_dir)

print("class:", type(student).__name__)
print("attrs:", [a for a in dir(student) if not a.startswith('__')])
print("has _model:", hasattr(student, '_model'), "  type:", type(getattr(student, '_model', None)).__name__)
print("has _tokenizer:", hasattr(student, '_tokenizer'), "  type:", type(getattr(student, '_tokenizer', None)).__name__)

# Find prompt-building method
import inspect
for name in dir(student):
    if 'prompt' in name.lower() or 'format' in name.lower() or 'input' in name.lower():
        obj = getattr(student, name)
        if callable(obj):
            try:
                sig = inspect.signature(obj)
                print(f"  {name}{sig}")
            except:
                print(f"  {name} (uninspectable)")

# Look at predict() source
print("\n--- predict() source ---")
try:
    print(inspect.getsource(student.predict))
except Exception as e:
    print(f"can't get source: {e}")
PY

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
python3 <<'PY'
import sys, json
sys.path.insert(0, '.')
import torch
from pathlib import Path
from src.phase2.config import load_phase2_config
from src.phase2.student import build_student, _format_input, _parse_output

cfg = load_phase2_config('configs/phase2_beta_qwen1.5b_lora_v4.yaml')
cfg.student.max_input_len  = 2048
cfg.student.max_target_len = 2048

student = build_student(cfg.student)
student.load(cfg.student_output_dir)
model, tok, device = student._model, student._tokenizer, student._device

# Same case that was REAL in smoke but EMPTY in eval
case = json.load(open('phase2_data/raw/echr/extracted/all-data/001-82175_5-3.json'))
args = case['all_arguments']
print(f"case has {len(args)} args\n")

input_A = ' '.join(a.get('text','') for a in args[:5])[:3000]   # smoke style
input_B = ' '.join(a.get('text','') for a in args)[:3000]       # eval style

for label, text in [('A_smoke_style_first5', input_A),
                    ('B_eval_style_allargs', input_B)]:
    print(f"\n{'='*72}\n{label}  input_len={len(text)}chars")
    print(f"  first 200: {text[:200]!r}")
    print(f"  last 200:  {text[-200:]!r}")

    inp = _format_input(text)
    prompt_str = tok.apply_chat_template(
        [{"role":"user","content":inp}], tokenize=False, add_generation_prompt=True)
    enc = tok([prompt_str], return_tensors='pt', truncation=True,
              max_length=cfg.student.max_input_len).to(device)
    print(f"  prompt_tokens={enc['input_ids'].shape[-1]}")

    with torch.no_grad():
        out = model.generate(**enc,
                             max_new_tokens=cfg.student.max_target_len,
                             pad_token_id=tok.pad_token_id,
                             eos_token_id=tok.eos_token_id,
                             do_sample=False)

    new_toks = out[0][enc['input_ids'].shape[-1]:]
    raw = tok.decode(new_toks, skip_special_tokens=True)
    print(f"  generated_tokens={new_toks.shape[0]}")
    print(f"  eos_reached={tok.eos_token_id in new_toks.tolist()}")
    print(f"\n  --- RAW (first 600 chars) ---\n{raw[:600]}")
    print(f"\n  --- RAW (last 400 chars) ---\n{raw[-400:] if len(raw) > 600 else '(same as above)'}")

    struct, reasoning = _parse_output(raw)
    n_c = len(struct.get('claim_components', []))
    n_p = len(struct.get('premise_components', []))
    print(f"\n  parsed: claims={n_c}  premises={n_p}  reasoning_chars={len(reasoning)}")
PY

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
mkdir -p eval_logs
nohup python3 -u <<'PY' > eval_logs/echr_v4_salvage.log 2>&1 &
import sys, json, random, time, re
sys.path.insert(0, '.')
import torch
from pathlib import Path
from src.phase2.config import load_phase2_config
from src.phase2.student import build_student, _format_input

# ── Lenient JSON salvage: scan raw output, pull valid {...} objs from each section ──
SECTIONS = [
    ('claim_components',   r'"claim_components"\s*:\s*\['),
    ('premise_components', r'"premise_components"\s*:\s*\['),
    ('citation_components',r'"citation_components"\s*:\s*\['),
    ('relations',          r'"relations"\s*:\s*\['),
]
def salvage_parse(raw):
    out = {k: [] for k, _ in SECTIONS}
    for key, pat in SECTIONS:
        m = re.search(pat, raw)
        if not m: continue
        start = m.end()
        depth, obj_start = 0, None
        for i in range(start, len(raw)):
            c = raw[i]
            if c == '{':
                if depth == 0: obj_start = i
                depth += 1
            elif c == '}':
                depth -= 1
                if depth == 0 and obj_start is not None:
                    s = raw[obj_start:i+1]
                    try:  out[key].append(json.loads(s))
                    except: pass
                    obj_start = None
            elif c == ']' and depth == 0: break
    # Dedupe relations
    seen, uniq = set(), []
    for r in out['relations']:
        k = (r.get('src'), r.get('tgt'), r.get('type'))
        if k not in seen: seen.add(k); uniq.append(r)
    out['relations'] = uniq
    return out

# ── Load model ──
cfg = load_phase2_config('configs/phase2_beta_qwen1.5b_lora_v4.yaml')
cfg.student.max_input_len  = 2048
cfg.student.max_target_len = 2048
student = build_student(cfg.student)
student.load(cfg.student_output_dir)
model, tok, dev = student._model, student._tokenizer, student._device
print("loaded v4 + salvage parser\n", flush=True)

# ── Sanity check on the case we already tested ──
case = json.load(open('phase2_data/raw/echr/extracted/all-data/001-82175_5-3.json'))
text = ' '.join(a.get('text','') for a in case['all_arguments'])[:3000]
inp = _format_input(text)
prompt = tok.apply_chat_template([{"role":"user","content":inp}], tokenize=False, add_generation_prompt=True)
enc = tok([prompt], return_tensors='pt', truncation=True, max_length=2048).to(dev)
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=2048, pad_token_id=tok.pad_token_id,
                         eos_token_id=tok.eos_token_id, do_sample=False)
raw = tok.decode(out[0][enc['input_ids'].shape[-1]:], skip_special_tokens=True)
s = salvage_parse(raw)
print(f"SANITY: case 001-82175_5-3 salvage → claims={len(s['claim_components'])} "
      f"premises={len(s['premise_components'])} relations={len(s['relations'])}\n", flush=True)

# ── ECHR eval with salvage ──
files = sorted(Path('phase2_data/raw/echr/extracted/all-data').glob('*.json'))
random.seed(42); sample = random.sample(files, 20)

def agent_to_role(a):
    if a == 'ECHR': return 'claim'
    if a in ('Applicant','State','Third Parties'): return 'premise'
    return None
def toks(s): return set(re.findall(r'\w+', s.lower()))
def overlap(a, b):
    ta, tb = toks(a), toks(b)
    return (len(ta & tb) / len(ta | tb)) if (ta and tb) else 0.0

tp_c=fp_c=fn_c=tp_p=fp_p=fn_p=n_real=n_ext=n_gtot=0
t0_all = time.time()
for idx, fp in enumerate(sample, 1):
    case = json.load(open(fp))
    args = case.get('all_arguments', [])
    gold = [{'text': a['text'], 'role': agent_to_role(a.get('agent'))}
            for a in args if agent_to_role(a.get('agent')) and a.get('text')]
    if not gold: continue
    text = ' '.join(a.get('text','') for a in args)[:3000]
    inp = _format_input(text)
    prompt = tok.apply_chat_template([{"role":"user","content":inp}], tokenize=False, add_generation_prompt=True)
    enc = tok([prompt], return_tensors='pt', truncation=True, max_length=2048).to(dev)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=2048, pad_token_id=tok.pad_token_id,
                             eos_token_id=tok.eos_token_id, do_sample=False)
    dt = time.time()-t0
    raw = tok.decode(out[0][enc['input_ids'].shape[-1]:], skip_special_tokens=True)
    struct = salvage_parse(raw)
    pred_spans = ([{'text':c.get('text',''),'role':'claim'} for c in struct['claim_components'] if c.get('text')] +
                  [{'text':p.get('text',''),'role':'premise'} for p in struct['premise_components'] if p.get('text')])
    gc = sum(1 for g in gold if g['role']=='claim'); gp = len(gold)-gc
    if not pred_spans:
        print(f"[{idx}/20] {fp.stem}: EMPTY t={dt:.0f}s gold_c/p={gc}/{gp}", flush=True)
        fn_c += gc; fn_p += gp; n_gtot += len(gold); continue
    mg, mp = [False]*len(gold), [False]*len(pred_spans)
    for pi, ps in enumerate(pred_spans):
        best_gi, best_ov = -1, 0.0
        for gi, gs in enumerate(gold):
            if mg[gi]: continue
            ov = overlap(ps['text'], gs['text'])
            if ov > best_ov: best_gi, best_ov = gi, ov
        if best_ov >= 0.5:
            mg[best_gi]=True; mp[pi]=True
            if ps['role']==gold[best_gi]['role']:
                if ps['role']=='claim': tp_c+=1
                else: tp_p+=1
            else:
                if ps['role']=='claim': fp_c+=1
                else: fp_p+=1
                if gold[best_gi]['role']=='claim': fn_c+=1
                else: fn_p+=1
    for pi,m in enumerate(mp):
        if not m:
            if pred_spans[pi]['role']=='claim': fp_c+=1
            else: fp_p+=1
    for gi,m in enumerate(mg):
        if not m:
            if gold[gi]['role']=='claim': fn_c+=1
            else: fn_p+=1
    n_ext += sum(mp); n_gtot += len(gold); n_real += 1
    pc = len(struct['claim_components']); pp = len(struct['premise_components'])
    print(f"[{idx}/20] {fp.stem}: pred_c/p={pc}/{pp} gold_c/p={gc}/{gp} t={dt:.0f}s", flush=True)

def f1(tp,fp,fn):
    p = tp/max(tp+fp,1); r = tp/max(tp+fn,1)
    return 2*p*r/max(p+r,1e-6), p, r
f1c,pc,rc = f1(tp_c,fp_c,fn_c); f1p,pp,rp = f1(tp_p,fp_p,fn_p)
print(f"\n=== ECHR w/ SALVAGE (non-empty {n_real}/20) ===")
print(f"claim   F1={f1c:.3f} P={pc:.3f} R={rc:.3f} (tp={tp_c} fp={fp_c} fn={fn_c})")
print(f"premise F1={f1p:.3f} P={pp:.3f} R={rp:.3f} (tp={tp_p} fp={fp_p} fn={fn_p})")
print(f"COMPONENT F1 (macro): {(f1c+f1p)/2:.3f}")
print(f"extraction rate: {n_ext}/{n_gtot} = {n_ext/max(n_gtot,1):.1%}")
print(f"wall: {(time.time()-t0_all)/60:.1f} min")
PY
echo "PID: $!"
sleep 2
echo "kicked off — log: eval_logs/echr_v4_salvage.log"
tail -5 eval_logs/echr_v4_salvage.log

In [ ]:
!cd ~/argument-aware-rag && git pull

In [ ]:
%%bash
cd ~/argument-aware-rag

# Everything currently in phase2_data
echo "=== phase2_data/raw/ ==="
ls -la phase2_data/raw/ 2>/dev/null

echo ""
echo "=== phase2_data/unified/ ==="
ls -la phase2_data/unified/ 2>/dev/null

echo ""
echo "=== phase2_data/silver/ ==="
ls -la phase2_data/silver/ 2>/dev/null

# Look for any other data directories
echo ""
echo "=== other data dirs ==="
find . -maxdepth 3 -type d -name "*data*" 2>/dev/null | head -30